# Hadoop, Hive, Spark 통합 실습

## 도커 실행 컨테이너 확인

In [1]:
! docker ps

CONTAINER ID   IMAGE                                     COMMAND                  CREATED             STATUS             PORTS                                                                    NAMES
181ff834c9dd   bde2020/hive:2.3.2-postgresql-metastore   "entrypoint.sh tail …"   About an hour ago   Up About an hour   10000/tcp, 10002/tcp                                                     hive-cli
ced7b097d695   bde2020/hive:2.3.2-postgresql-metastore   "entrypoint.sh /bin/…"   About an hour ago   Up About an hour   0.0.0.0:10000->10000/tcp, 10002/tcp                                      hive-server
d4eaca873f09   confluentinc/cp-kafka:7.6.0               "/etc/confluent/dock…"   About an hour ago   Up About an hour   0.0.0.0:9092->9092/tcp, 0.0.0.0:19092->19092/tcp                         kafka
74704c5b0c8f   bde2020/hive:2.3.2-postgresql-metastore   "entrypoint.sh /opt/…"   About an hour ago   Up About an hour   10000/tcp, 0.0.0.0:9083->9083/tcp, 10002/tcp                          

## 도커 볼륨 확인

<pre> <font color=red><b>docker-compose.yml, Dockerfile.hadoop</b></font>
- volumes:
      - ./config/hadoop:/root/config   #로컬 마운트
      - hadoop_data:/hadoop

| 항목    | 실제 호스트 경로| 컨테이너 내부 경로                                        |
| ------ | -------------- | ------------------------------------------- |
| 로컬 마운트  | `C:\DC\t3-hadoop-hive-spark\config\hadoop`  | `/root/config`                 |
| 도커 볼륨   | ` hadoop_data (/var/lib/docker/volumes/t3-hadoop-hive-spark_hadoop_data/_data) ` | `/hadoop`  |


In [9]:
! docker volume ls

DRIVER    VOLUME NAME
local     5ee62af6ba462084d2ccbd992580db0962c7fa9b9d4048a9b044ac622f43ca31
local     0885c773ff2f06fc347b9c09279cd66110a223dec7bb305c2953e28a269e9b4f
local     3283c9469c94f9efbbba75605bb831cba6704b72b2ff1b54d84389b8defebab2
local     aa871063e0776042c74a877416fb1244b085d00eea33066a4dd0fcfc71a15912
local     d4f9d48c9642066073a420d4649605367645a2e8cb3621edb3000ae4015f4167
local     t3-hadoop-hive-spark_hadoop_data
local     t4-hive-spark_pgdata


In [10]:
! docker volume inspect t3-hadoop-hive-spark_hadoop_data

[
    {
        "CreatedAt": "2025-06-28T14:50:24Z",
        "Driver": "local",
        "Labels": {
            "com.docker.compose.config-hash": "a95c1825460545e2f882f6310530d4a670ffa6d207033388fa724c36042959c0",
            "com.docker.compose.project": "t3-hadoop-hive-spark",
            "com.docker.compose.version": "2.35.1",
            "com.docker.compose.volume": "hadoop_data"
        },
        "Mountpoint": "/var/lib/docker/volumes/t3-hadoop-hive-spark_hadoop_data/_data",
        "Name": "t3-hadoop-hive-spark_hadoop_data",
        "Options": null,
        "Scope": "local"
    }
]


## 하둡 네트워크 확인

<pre> <font color=red><b>docker-compose.yml</b></font>
networks:
  hive-net:
    name: hive-net
    driver: bridge
  hadoop:
    name: hadoop
    driver: bridge

In [14]:
! docker network ls

NETWORK ID     NAME                    DRIVER    SCOPE
9169974b990e   bridge                  bridge    local
d9b820719876   hadoop                  bridge    local
1aaa4d216cbe   hive-net                bridge    local
f05d281638da   host                    host      local
2acf21e3fa18   none                    null      local
3254e4a3f11a   t1-hadoop_default       bridge    local
f77f84560f76   t1-hadoop_hadoopnet     bridge    local
1916d1b2de9c   t4-hive-spark_default   bridge    local


In [19]:
! docker inspect hadoop-master| grep "IPAddress"

            "SecondaryIPAddresses": null,
            "IPAddress": "",
                    "IPAddress": "172.21.0.4",


In [20]:
! docker inspect hadoop-slave| grep "IPAddress"

            "SecondaryIPAddresses": null,
            "IPAddress": "",
                    "IPAddress": "172.21.0.2",


## 하둡 네트워크와 연결된 컨테이너 확인
- hadoop-master: 172.21.0.4/16
- hadoop-slave: 172.21.0.2/16
- hive-metastore : 172.21.0.5/16
- spark-master : 172.21.0.3/16
- spark-worker : 172.21.0.6/16
- spark-client: 172.21.0.7/16

In [21]:
! docker network inspect hadoop

[
    {
        "Name": "hadoop",
        "Id": "d9b820719876d53c0ecf819e116e4a541eb6dc56c519d4c1e3df2d6ee91fbd14",
        "Created": "2025-06-28T16:26:27.965202637Z",
        "Scope": "local",
        "Driver": "bridge",
        "EnableIPv4": true,
        "EnableIPv6": false,
        "IPAM": {
            "Driver": "default",
            "Options": null,
            "Config": [
                {
                    "Subnet": "172.21.0.0/16",
                    "Gateway": "172.21.0.1"
                }
            ]
        },
        "Internal": false,
        "Attachable": false,
        "Ingress": false,
        "ConfigFrom": {
            "Network": ""
        },
        "ConfigOnly": false,
        "Containers": {
            "017dec2e6e5ad5aed7887706d749c9edcdf5b7479dee266b81eb4bf2e8b941d8": {
                "Name": "hive-metastore",
                "EndpointID": "abc033a04a82d5793df0527534733e2fd77a942ab773d3e5a2b7c7ad617670c8",
                "MacAddress": "9a:86:0f:c1:c9:

# **하둡 컨테이너 내부 진입**

## 하둡 버전 확인

In [12]:
! docker exec -it hadoop-master sh -c  'hadoop version'

Hadoop 3.3.6
Source code repository https://github.com/apache/hadoop.git -r 1be78238728da9266a4f88195058f08fd012bf9c
Compiled by ubuntu on 2023-06-18T08:22Z
Compiled on platform linux-x86_64
Compiled with protoc 3.7.1
From source with checksum 5652179ad55f76cb287d9c633bb53bbd
This command was run using /opt/hadoop/share/hadoop/common/hadoop-common-3.3.6.jar


## 하둡 홈디렉토리 확인

In [5]:
!docker exec -it hadoop-master sh -c 'cat $HADOOP_HOME'

cat: /opt/hadoop: Is a directory


## 현재 작업 디렉토리 확인

In [7]:
! docker exec -it hadoop-master sh -c   'pwd'

/


In [8]:
! pwd

/opt/bitnami/spark/workspace_pyspark


In [13]:
!docker exec -it hadoop-master sh -c 'ls -l /opt/hadoop'

total 116
-rw-rw-r-- 1 1000 1000 24276 Jun 14  2023 LICENSE-binary
-rw-rw-r-- 1 1000 1000 15217 Jun  9  2023 LICENSE.txt
-rw-rw-r-- 1 1000 1000 29473 Jun  9  2023 NOTICE-binary
-rw-rw-r-- 1 1000 1000  1541 Jun  9  2023 NOTICE.txt
-rw-rw-r-- 1 1000 1000   175 Jun  9  2023 README.txt
drwxr-xr-x 2 1000 1000  4096 Jun 18  2023 bin
drwxr-xr-x 1 1000 1000  4096 Jun 18  2023 etc
drwxr-xr-x 2 1000 1000  4096 Jun 18  2023 include
drwxr-xr-x 3 1000 1000  4096 Jun 18  2023 lib
drwxr-xr-x 4 1000 1000  4096 Jun 18  2023 libexec
drwxr-xr-x 2 1000 1000  4096 Jun 18  2023 licenses-binary
drwxr-xr-x 3 root root  4096 Jun 28 16:26 logs
drwxr-xr-x 3 1000 1000  4096 Jun 18  2023 sbin
drwxr-xr-x 4 1000 1000  4096 Jun 18  2023 share


## 하둡 HDFS 기본 URI

* <font color=red><b>C:\DC\t3-hadoop-hive-spark\config\hadoop\core-site.xml

In [22]:
!docker exec -it hadoop-master sh -c 'hdfs getconf -confKey fs.defaultFS'

hdfs://hadoop-master:9000


## 하둡 관리 페이지

* HDFS UI(NameNode UI)
    - http://localhost:9870 
* YARN UI(ResourceManager UI)
    - http://localhost:8088 

# [실습]

## 1. 데이터 압축 해제 및 HDFS 업로드

In [2]:
!unzip -o t3-hadoop-hive-spark-20250618T064223Z-1-001.zip -d data/
# HDFS 업로드
! docker exec -it hadoop-master sh -c  'hdfs dfs -mkdir -p /user/hduser/input
!hdfs dfs -put -f data/*.csv /user/hduser/input/   
#csv파일업로드

/bin/sh: 1: unzip: not found
/bin/sh: 1: hdfs: not found


## 2. Hadoop MapReduce 실습

In [ ]:
%%writefile mapper.pyimport sysimport csvreader = csv.reader(sys.stdin)next(reader, None)  # 헤더 제거for row in reader:    print(f"{row[2]}	1")  # 예시: category 컬럼%%writefile reducer.pyimport syscurrent_key = Nonecurrent_count = 0for line in sys.stdin:    key, value = line.strip().split('\t')    value = int(value)    if key == current_key:        current_count += value    else:        if current_key:            print(f"{current_key}\t{current_count}")        current_key = key        current_count = valueif current_key:    print(f"{current_key}\t{current_count}")

In [ ]:
!hadoop jar $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming*.jar \    -input /user/hduser/input/*.csv \    -output /user/hduser/output1 \    -mapper mapper.py \    -reducer reducer.py \    -file mapper.py \    -file reducer.py

## 3. Hive 테이블 생성 및 쿼리

In [ ]:
!hive -e "CREATE EXTERNAL TABLE IF NOT EXISTS sales_data (  id INT,  product STRING,  category STRING,  amount DOUBLE)ROW FORMAT DELIMITEDFIELDS TERMINATED BY ','STORED AS TEXTFILELOCATION '/user/hduser/input/';"!hive -e "SELECT category, COUNT(*) FROM sales_data GROUP BY category;"

## 4. Spark + Hive 분석

In [ ]:
from pyspark.sql import SparkSessionspark = SparkSession.builder \    .appName("SparkHiveExample") \    .enableHiveSupport() \    .getOrCreate()df = spark.sql("SELECT category, COUNT(*) as cnt FROM sales_data GROUP BY category")df.show()

## 5. Hadoop + Spark + Hive 통합

In [ ]:
df2 = spark.read.option("header", True).csv("hdfs:///user/hduser/input/*.csv")df2.createOrReplaceTempView("raw_data")spark.sql("DROP TABLE IF EXISTS sales_temp")spark.sql("CREATE TABLE sales_temp AS SELECT * FROM raw_data")spark.sql("SELECT category, AVG(amount) FROM sales_temp GROUP BY category").show()

## 6. Spark 결과를 MySQL로 저장

In [ ]:
df2.write \    .format("jdbc") \    .option("url", "jdbc:mysql://localhost:3306/mydb") \    .option("dbtable", "category_avg") \    .option("user", "root") \    .option("password", "1234") \    .save()